# Assignment 5 — Many-Objective Optimisation with JUSTICE (Local Run)

**Course:** EPA141A Model-Based Decision Making — Delft University of Technology  
**Model:** JUSTICE  

---

## Learning Outcomes

After completing this assignment you will be able to:
1. Load and inspect an optimisation configuration from a JSON file produced in Assignment 4.
2. Run a many-objective evolutionary algorithm (**GenerationalBorg**) on the JUSTICE model via `run_optimization_local.py`.
3. Load and inspect the Pareto-front CSV files produced by a completed MOEA run.

---

## Background

Assignment 4 defined the optimisation problem: 244 RBF decision variables, four objectives, and epsilon values that control archive resolution. This assignment runs the actual Pareto search.

The optimiser is **GenerationalBorg**, a self-adaptive MOEA that simultaneously applies six variation operators (SBX, PCX, DE, UNDX, SPX, UM) and continuously adjusts their selection probabilities based on which generates the most archive improvement. This makes it well suited to high-dimensional spaces like the 244-parameter RBF search space.

Each function evaluation runs the JUSTICE model once with one candidate policy (one set of 244 RBF parameters), returning four objective values. The algorithm accumulates non-dominated solutions in an **epsilon-archive**: a candidate is only archived if it improves on the current archive by at least ε in at least one objective. Smaller ε gives finer Pareto resolution but requires more function evaluations.

Because MOEAs are stochastic, a single run may miss parts of the Pareto front. Running five independent seeds and pooling their archives with epsilon-dominance produces a **combined reference set** that is more complete than any individual seed. This reference set is the starting point for the Pareto and robustness analyses in Assignments 6 and 7.

---

## Starting Point — Your Assignment 4 configuration

In Assignment 4 you defined the multi-objective optimisation problem and saved your choices to `config/config_student.json`. This notebook loads that config and runs the optimisation using `run_optimization_local.py`.

**Run commands** (execute in a terminal from any directory):

| Mode | Command |
|---|---|
| Smoke test (< 5 min) | `python run_optimization_local.py --nfe 500 --seeds 9845531 --config config/config_student.json` |
| Single (~3 h) | `python run_optimization_local.py --nfe 50000 --seeds 9845531 --config config/config_student.json` |
| Full 5-seed run (overnight) | `python run_optimization_local.py --nfe 50000 --config config/config_student.json` |

---

## Step 1 — Inspect the Configuration

The optimisation is governed by `config/config_student.json`.  
Read and print each key with an explanation.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, sys, json
import numpy as np
import pandas as pd

# Locate JUSTICE-main (one level up from model_answers_ema/)
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

# ── Inspect config ────────────────────────────────────────────────────────────
CONFIG_PATH = os.path.join(_NOTEBOOK_DIR, "../config/config_student.json")

with open(CONFIG_PATH) as fh:
    cfg = json.load(fh)

explanations = {
    "start_year":                       "Simulation start year",
    "end_year":                         "Simulation end year",
    "data_timestep":                    "Years between raw input data points",
    "timestep":                         "Model integration timestep (years)",
    "emission_control_start_year":      "First year ECR can exceed zero",
    "n_rbfs":                           "Number of RBFs (effective: n_inputs + 2)",
    "n_inputs":                         "RBF input signals (scaled temperature + Δtemperature)",
    "epsilons":                         "Archive granularity [welfare, frac_above, wl_damage, wl_abatement]",
    "temperature_year_of_interest":     "Year for threshold fraction evaluation",
    "reference_ssp_rcp_scenario_index": "Reference scenario index (2 = SSP2-RCP4.5)",
}

print(f"Configuration: {CONFIG_PATH}\n")
for k, v in cfg.items():
    print(f"  {k:<40s}  {str(v):<15}  # {explanations.get(k, '')}")

Configuration: /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/../config/config_student.json

  start_year                                2015             # Simulation start year
  end_year                                  2300             # Simulation end year
  data_timestep                             5                # Years between raw input data points
  timestep                                  1                # Model integration timestep (years)
  emission_control_start_year               2025             # First year ECR can exceed zero
  n_rbfs                                    4                # Number of RBFs (effective: n_inputs + 2)
  n_inputs                                  2                # RBF input signals (scaled temperature + Δtemperature)
  epsilons                                  [0.1, 0.25, 10.0, 10.0]  # Archive granularity [welfare, frac_above, wl_damage, wl_abatement]
  temperature_yea

In [2]:
from justice.util.data_loader import DataLoader

# Decision variable count
n_inputs   = cfg["n_inputs"]
n_rbfs_act = n_inputs + 2
n_regions  = len(DataLoader().REGION_LIST)

n_centers = n_rbfs_act * n_inputs
n_radii   = n_rbfs_act * n_inputs
n_weights = n_rbfs_act * n_regions
n_total   = n_centers + n_radii + n_weights

print("RBF decision variable summary:")
print(f"  n_rbfs_actual = {n_rbfs_act}  (n_inputs + 2 = {n_inputs} + 2)")
print(f"  n_regions     = {n_regions}")
print(f"  ─────────────────────────────────")
print(f"  Centers       = {n_rbfs_act} × {n_inputs} = {n_centers:4d}  range [-1,  1]")
print(f"  Radii         = {n_rbfs_act} × {n_inputs} = {n_radii:4d}  range [ 0,  1]")
print(f"  Weights       = {n_rbfs_act} × {n_regions} = {n_weights:4d}  range [ 0,  1]")
print(f"  ─────────────────────────────────")
print(f"  TOTAL                    = {n_total}")

print(f"\nEpsilon values per objective:")
obj_names = ["welfare", "fraction_above_threshold", "welfare_loss_damage", "welfare_loss_abatement"]
for name, eps in zip(obj_names, cfg["epsilons"]):
    print(f"  {name:<35s}  ε = {eps}")

RBF decision variable summary:
  n_rbfs_actual = 4  (n_inputs + 2 = 2 + 2)
  n_regions     = 57
  ─────────────────────────────────
  Centers       = 4 × 2 =    8  range [-1,  1]
  Radii         = 4 × 2 =    8  range [ 0,  1]
  Weights       = 4 × 57 =  228  range [ 0,  1]
  ─────────────────────────────────
  TOTAL                    = 244

Epsilon values per objective:
  welfare                              ε = 0.1
  fraction_above_threshold             ε = 0.25
  welfare_loss_damage                  ε = 10.0
  welfare_loss_abatement               ε = 10.0


---

## Step 2 — Run the Optimisation

The optimisation is implemented in `run_optimization_local.py`.  
Open a **terminal** in the `model_answers_ema/` directory and run:

```bash
# Quick smoke-test (~3-5 min, 1 seed, 500 NFE)
python run_optimization_local.py --nfe 500 --seeds 9845531

# Full single-seed run (~1-3 h)
python run_optimization_local.py --nfe 50000 --seeds 9845531

# Full 5-seed run (background, ~1 working day)
nohup python run_optimization_local.py --nfe 50000 > opt_log.txt 2>&1 &
```

You can also launch it from this notebook cell (blocks the kernel until finished):

In [3]:
# ── CONFIGURE HERE ───────────────────────────────────────────────────────────
NFE         = 500          # Use 500 for a smoke-test; 50_000 for the full run
SEEDS       = [9845531]    # All 5 seeds: [9845531, 1644652, 3569126, 6075612, 521475]
N_ENSEMBLES = 10           # 10 well-distributed FAIR members (local speed)
N_PROCESSES = None         # None → auto-detect (cpu_count - 1)
OUTPUT_DIR  = os.path.join(_NOTEBOOK_DIR, "results")
# ─────────────────────────────────────────────────────────────────────────────

seeds_str  = " ".join(str(s) for s in SEEDS)
n_proc_arg = f"--n_processes {N_PROCESSES}" if N_PROCESSES else ""

# Derive script path from _JUSTICE_ROOT (reliable regardless of kernel CWD)
script_path = os.path.normpath(
    os.path.join(_JUSTICE_ROOT, "..", "model_answers_ema", "run_optimization_local.py")
)
cmd = (
    f"python {script_path} "
    f"--nfe {NFE} "
    f"--seeds {seeds_str} "
    f"--n_ensembles {N_ENSEMBLES} "
    f"--output_dir {OUTPUT_DIR} "
    f"--config " + os.path.join(_NOTEBOOK_DIR, "../config/config_student.json") + " "
    + n_proc_arg
)

print("Command to run:")
print(f"  {cmd}")
print("\nRunning ... (this cell blocks until complete)")

ret = os.system(cmd)
print(f"\nExit code: {ret}  ({'OK' if ret == 0 else 'ERROR'})")

Command to run:
  python /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/run_optimization_local.py --nfe 500 --seeds 9845531 --n_ensembles 10 --output_dir /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/results --config /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/../config/config_student.json 

Running ... (this cell blocks until complete)


JUSTICE — Local MOEA Optimisation  (Assignment 5)
  Welfare function  : UTILITARIAN
  Config            : /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/../config/config_student.json
  NFE per seed      : 500
  Seeds (1)        : [9845531]
  FAIR ensembles    : 10  (indices ≈ [np.int64(1), np.int64(112), np.int64(223), np.int64(334), np.int64(445), np.int64(556), np.int64(667), np.int64(778), np.int64(889), np.int64(1000)])
  Population size   : 100
  Worker processes  : auto
  Output directory  : /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/results

[1/1] seed = 9845531
----------------------------------------
    removed stale archive: /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/results/UTILITARIAN_500_9845531/UTILITARIAN_500_9845531.tar.gz
    workers = auto (cpu_co

[MainProcess/INFO] pool started with 14 workers
  0%|                                                  | 0/500 [00:00<?, ?it/s]

 20%|████████                                | 100/500 [00:03<00:14, 27.82it/s]

 20%|████████                                | 100/500 [00:20<00:14, 27.82it/s]

 40%|████████████████                        | 200/500 [00:23<00:40,  7.47it/s]

 60%|████████████████████████                | 301/500 [00:41<00:30,  6.54it/s]

 80%|████████████████████████████████▏       | 402/500 [01:03<00:17,  5.60it/s]

503it [01:24,  5.98it/s]
[MainProcess/INFO] optimization completed, found 8 solutions
[MainProcess/INFO] terminating pool


    convergence metrics  →  /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/results/UTILITARIAN_500_9845531/convergence_9845531.csv
    8 Pareto solutions  →  /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/results/UTILITARIAN_500_9845531/pareto_front_9845531.csv
    convergence archive  →  /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/results/UTILITARIAN_500_9845531/UTILITARIAN_500_9845531.tar.gz
    elapsed: 1.7 min

All 1 seeds done in 2 min (0.03 h).
Results → /Users/jzatarinsalaza/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/epa141a-MBDM/epa141a/model_answers_ema/results

Exit code: 0  (OK)


---

## Step 3 — Load and Inspect Results

Each completed seed produces a directory `UTILITARIAN_<nfe>_<seed>/` inside `results/` containing:
- `pareto_front_<seed>.csv` — the final Pareto-optimal solutions (levers + objectives).
- `UTILITARIAN_<nfe>_<seed>.tar.gz` — the ArchiveLogger convergence history (used in Assignment 6).
- `convergence_<seed>.csv` — EpsilonProgress and operator probabilities per NFE checkpoint (used in Assignment 6).

Load all available Pareto front CSVs and print a statistical summary.

In [4]:
import glob

# Path to results (model_answers_ema/results/)
RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))

OBJECTIVE_COLS = [
    "welfare",
    "fraction_above_threshold",
    "welfare_loss_damage",
    "welfare_loss_abatement",
]

csv_files = sorted(glob.glob(
    os.path.join(RESULTS_ROOT, "**", "pareto_front_*.csv"), recursive=True
))

if not csv_files:
    print(f"No results found in {RESULTS_ROOT}.")
    print("Run Step 2 first, then re-run this cell.")
else:
    dfs = []
    for path in csv_files:
        seed_str = os.path.basename(path).replace("pareto_front_", "").replace(".csv", "")
        # Extract NFE from parent directory name: UTILITARIAN_<nfe>_<seed>
        dir_name = os.path.basename(os.path.dirname(path))
        parts    = dir_name.split("_")
        try:
            nfe = int(parts[-2])
        except (ValueError, IndexError):
            nfe = None
        df = pd.read_csv(path)
        df["seed"] = int(seed_str)
        df["nfe"]  = nfe
        dfs.append(df)
        nfe_str = f"{nfe:,}" if nfe else "?"
        print(f"  seed {seed_str:>10s}  NFE {nfe_str:>8s}: {len(df):4d} Pareto solutions")

    all_results = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal: {len(all_results)} solutions across {len(dfs)} seed(s).")
    print(f"\nObjective statistics:")
    print(all_results[OBJECTIVE_COLS].describe().round(4))


  seed    1644652  NFE   50,000:   14 Pareto solutions
  seed    3569126  NFE   50,000:   11 Pareto solutions
  seed     521475  NFE   50,000:   10 Pareto solutions
  seed    6075612  NFE   50,000:   10 Pareto solutions
  seed    9845531  NFE   50,000:    9 Pareto solutions
  seed    1644652  NFE      500:    6 Pareto solutions
  seed    3569126  NFE      500:    6 Pareto solutions
  seed     521475  NFE      500:    8 Pareto solutions
  seed    6075612  NFE      500:    6 Pareto solutions
  seed    9845531  NFE      500:    8 Pareto solutions

Total: 88 solutions across 10 seed(s).

Objective statistics:
            welfare  fraction_above_threshold  welfare_loss_damage  \
count       88.0000                   88.0000              88.0000   
mean     56915.8301                    0.6875            3428.2951   
std     232797.3975                    0.3307             846.3224   
min        103.4492                    0.0000               0.0000   
25%        103.4681                  

In [5]:
# Verify results are non-trivial:
# A non-trivial Pareto front should have:
#   - More than 1 solution
#   - Some variation in all four objective columns
#   - fraction_above_threshold < 1.0 for at least some solutions
#     (meaning the MOEA found policies better than BAU)

try:
    all_results
except NameError:
    print("Run the previous cell first.")
    raise

obj = all_results[OBJECTIVE_COLS]

checks = [
    ("More than 1 Pareto solution",
     len(obj) > 1),
    ("No NaN values in objectives",
     obj.notna().all().all()),
    ("fraction_above_threshold spans > 0.05",
     obj["fraction_above_threshold"].max() - obj["fraction_above_threshold"].min() > 0.05),
    ("welfare_loss_abatement spans > 0",
     obj["welfare_loss_abatement"].std() > 0),
    ("At least one solution with frac_above < BAU (frac=1.0)",
     (obj["fraction_above_threshold"] < 1.0).any()),
]

print("Non-triviality checks:")
for label, ok in checks:
    print(f"  {'✓' if ok else '✗'}  {label}")

passed = sum(ok for _, ok in checks)
print(f"\n{passed}/{len(checks)} checks passed.")
if passed == len(checks):
    print("Results look healthy — proceed to Assignment 6 for analysis.")
else:
    print("Some checks failed — consider increasing NFE or verifying the script ran correctly.")

Non-triviality checks:
  ✓  More than 1 Pareto solution
  ✓  No NaN values in objectives
  ✓  fraction_above_threshold spans > 0.05
  ✓  welfare_loss_abatement spans > 0
  ✓  At least one solution with frac_above < BAU (frac=1.0)

5/5 checks passed.
Results look healthy — proceed to Assignment 6 for analysis.


---

## Reflection Questions

**1. Multiple seeds.** Why do we run the MOEA with 5 different random seeds rather than running a single long seed for 5× the NFE? What property of the algorithm makes this necessary?

> MOEAs are stochastic — different random seeds explore different regions of the 244-dimensional search space and may converge to different parts of the Pareto front. A single long run can get stuck near a local optimum. Pooling 5 independent seeds and applying epsilon-dominance gives a combined reference set that is more complete than any individual seed. GenerationalBorg's self-adaptive operator selection also starts fresh each seed, so different seeds try different operator mixes.

**2. NFE size.**  What diagnostic would tell you whether the NFE you tried was enough?

> The convergence plots (to be developed in Assignment 6) specifically the hypervolume and epsilon-progress curves, would show whether improvement had plateaued well before NFE was exhausted. If the curves are still rising at X NFE, the budget was insufficient.


**3. Operator adaptation.** GenerationalBorg continuously adapts the selection probabilities of its six variation operators (SBX, PCX, DE, UNDX, SPX, UM) during the run. What does this mean and why is it useful for the 244-dimensional RBF search space?

> At the start of the run all operators are equally likely to be selected. After each generation the algorithm records which operators produced solutions that improved the archive, and increases their selection probability. For a 244-dimensional space no single operator is universally best — SBX works well near the current archive, DE handles correlated variables, and UM introduces diversity when the search stagnates. Automatic adaptation means the algorithm self-tunes to the geometry of the specific problem rather than requiring the user to pre-select operators manually.

---

## Appendix — Script Reference

```
run_optimization_local.py — full argument list

  --nfe           int     NFE per seed                 (default: 50000)
  --seeds         int+    Random seeds                 (default: all 5)
  --output_dir    str     Results root directory        (default: model_answers_ema/results/)
  --n_processes   int     Worker processes              (default: auto)
  --n_ensembles   int     FAIR ensemble members         (default: 10)
  --config        str     JSON config path              (default: config/config_student.json)
  --population    int     MOEA population size          (default: 100)
```

**Output directory structure:**

```
model_answers_ema/
└── results/
    └── UTILITARIAN_<nfe>_<seed>/
        ├── UTILITARIAN_<nfe>_<seed>.tar.gz   ← convergence archive (Assignment 6)
        ├── convergence_<seed>.csv            ← EpsilonProgress + operator probs (Assignment 6)
        └── pareto_front_<seed>.csv           ← final Pareto front
```

For example, running with `--nfe 50000 --seeds 9845531` produces:

```
model_answers_ema/
└── results/
    └── UTILITARIAN_50000_9845531/
        ├── UTILITARIAN_50000_9845531.tar.gz
        ├── convergence_9845531.csv
        └── pareto_front_9845531.csv
```